# Weather EDA and Prophet baseline

This notebook evaluates one sufficiently covered district at a time. It uses monthly district weather data, preserves missing observations, and compares Prophet against a seasonal-naive baseline.

Run the normalization, district-resolution, and district-aggregation scripts before this notebook. Humidity is intentionally excluded because the supplied historical files do not contain humidity observations.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

In [ ]:
DATA_CANDIDATES = [
    Path('data/weather/processed/district_weather_monthly.csv'),
    Path('../../data/weather/processed/district_weather_monthly.csv'),
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), DATA_CANDIDATES[0])
RESULTS_PATH = DATA_PATH.parent / 'baseline_model_results.csv'
TEST_MONTHS = 12
MIN_TRAIN_MONTHS = 24
MIN_COVERAGE_MONTHS = 48

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'{DATA_PATH} was not found. Run district resolution and '
        'district aggregation before opening this notebook.'
    )

In [ ]:
weather = pd.read_csv(DATA_PATH, parse_dates=['observation_date'])
required_columns = {
    'district_id', 'observation_date', 'rainfall_mm',
    'temperature_min_c', 'temperature_max_c', 'temperature_avg_c',
    'station_count',
}
missing_columns = required_columns - set(weather.columns)
if missing_columns:
    raise ValueError(f'District dataset is missing columns: {sorted(missing_columns)}')

weather = weather.sort_values(['district_id', 'observation_date']).reset_index(drop=True)
print('Shape:', weather.shape)
print('Date range:', weather['observation_date'].min(), 'to', weather['observation_date'].max())
print('Districts:', weather['district_id'].nunique())
display(weather.head())

In [ ]:
display(weather.describe(include='all'))

missing_summary = weather.isna().sum().to_frame('missing_count')
missing_summary['missing_percentage'] = (
    missing_summary['missing_count'] / len(weather) * 100
).round(2)
display(missing_summary.sort_values('missing_count', ascending=False))

# Missing values remain missing: zero rainfall is a valid observation.

In [ ]:
group_columns = ['district_id']
if 'district_name' in weather.columns:
    group_columns.append('district_name')

coverage = (
    weather.groupby(group_columns, dropna=False)
    .agg(
        start_date=('observation_date', 'min'),
        end_date=('observation_date', 'max'),
        total_months=('observation_date', 'nunique'),
        rainfall_months=('rainfall_mm', 'count'),
        temperature_months=('temperature_avg_c', 'count'),
        avg_station_count=('station_count', 'mean'),
    )
    .reset_index()
)
coverage['rainfall_trainable'] = coverage['rainfall_months'] >= MIN_COVERAGE_MONTHS
coverage['temperature_trainable'] = coverage['temperature_months'] >= MIN_COVERAGE_MONTHS
coverage = coverage.sort_values(['rainfall_months', 'temperature_months'], ascending=False)
display(coverage)

In [ ]:
eligible = coverage[
    coverage['rainfall_trainable'] & coverage['temperature_trainable']
].copy()
if eligible.empty:
    raise ValueError(
        'No district has at least 48 monthly rainfall and average-temperature observations. '
        'Review the coverage table instead of forcing a model.'
    )

EXPERIMENT_DISTRICT_ID = int(eligible.iloc[0]['district_id'])
print('Selected district:', EXPERIMENT_DISTRICT_ID)
display(eligible.head())

district_weather = (
    weather.loc[weather['district_id'] == EXPERIMENT_DISTRICT_ID]
    .copy()
    .sort_values('observation_date')
    .reset_index(drop=True)
)
display(district_weather)

In [ ]:
expected_dates = pd.date_range(
    district_weather['observation_date'].min(),
    district_weather['observation_date'].max(),
    freq='MS',
)
missing_dates = expected_dates.difference(district_weather['observation_date'])
print('Missing calendar months:', len(missing_dates))
display(pd.DataFrame({'missing_month': missing_dates}))

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
axes[0].plot(district_weather['observation_date'], district_weather['rainfall_mm'], marker='o')
axes[0].set(title='Monthly rainfall', ylabel='Rainfall (mm)')
axes[0].grid(alpha=0.3)
axes[1].plot(district_weather['observation_date'], district_weather['temperature_min_c'], label='Minimum')
axes[1].plot(district_weather['observation_date'], district_weather['temperature_avg_c'], label='Average')
axes[1].plot(district_weather['observation_date'], district_weather['temperature_max_c'], label='Maximum')
axes[1].set(title='Monthly temperature', xlabel='Month', ylabel='Temperature (C)')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
seasonality = district_weather.assign(month=district_weather['observation_date'].dt.month)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
seasonality.groupby('month')['rainfall_mm'].mean().plot(kind='bar', ax=axes[0])
axes[0].set(title='Average rainfall by month', xlabel='Month', ylabel='Rainfall (mm)')
seasonality.groupby('month')['temperature_avg_c'].mean().plot(kind='bar', ax=axes[1])
axes[1].set(title='Average temperature by month', xlabel='Month', ylabel='Temperature (C)')
plt.tight_layout()
plt.show()

In [ ]:
def prepare_target(data: pd.DataFrame, target: str) -> pd.DataFrame:
    result = (
        data[['observation_date', target]]
        .rename(columns={'observation_date': 'ds', target: 'y'})
        .dropna()
        .sort_values('ds')
        .drop_duplicates('ds', keep='last')
        .reset_index(drop=True)
    )
    if len(result) < MIN_TRAIN_MONTHS + TEST_MONTHS:
        raise ValueError(f'{target} has insufficient observations for this experiment.')
    return result

def seasonal_naive_forecast(train: pd.DataFrame, test: pd.DataFrame) -> np.ndarray:
    history = train.set_index('ds')['y'].to_dict()
    predictions = []
    for date in test['ds']:
        previous_year = date - pd.DateOffset(years=1)
        prediction = history.get(previous_year, np.nan)
        predictions.append(prediction)
    return np.asarray(predictions, dtype=float)

def evaluate_forecast(y_true, y_pred) -> dict[str, float]:
    actual = np.asarray(y_true, dtype=float)
    predicted = np.asarray(y_pred, dtype=float)
    usable = np.isfinite(actual) & np.isfinite(predicted)
    if not usable.any():
        return {'MAE': np.nan, 'RMSE': np.nan, 'MAPE': np.nan, 'n': 0}
    actual, predicted = actual[usable], predicted[usable]
    nonzero = np.abs(actual) > 1e-8
    mape = np.mean(np.abs((actual[nonzero] - predicted[nonzero]) / actual[nonzero])) * 100 if nonzero.any() else np.nan
    return {
        'MAE': mean_absolute_error(actual, predicted),
        'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
        'MAPE': mape,
        'n': int(len(actual)),
    }

def prophet_forecast(train: pd.DataFrame, test: pd.DataFrame, clip_zero: bool) -> np.ndarray:
    model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    model.fit(train)
    forecast = model.predict(test[['ds']])['yhat'].to_numpy()
    return np.clip(forecast, 0, None) if clip_zero else forecast

def run_experiment(data: pd.DataFrame, target: str, clip_zero: bool) -> tuple[pd.DataFrame, pd.DataFrame]:
    series = prepare_target(data, target)
    train, test = series.iloc[:-TEST_MONTHS].copy(), series.iloc[-TEST_MONTHS:].copy()
    naive = seasonal_naive_forecast(train, test)
    prophet = prophet_forecast(train, test, clip_zero)
    comparison = pd.DataFrame({'date': test['ds'].to_numpy(), 'actual': test['y'].to_numpy(), 'seasonal_naive': naive, 'prophet': prophet})
    metrics = pd.DataFrame([
        {'model': 'Seasonal Naive', **evaluate_forecast(test['y'], naive)},
        {'model': 'Prophet', **evaluate_forecast(test['y'], prophet)},
    ])
    return comparison, metrics

In [ ]:
rainfall_comparison, rainfall_metrics = run_experiment(district_weather, 'rainfall_mm', clip_zero=True)
display(rainfall_metrics)
display(rainfall_comparison)

plt.figure(figsize=(14, 5))
for column, label in [('actual', 'Actual'), ('seasonal_naive', 'Seasonal Naive'), ('prophet', 'Prophet')]:
    plt.plot(rainfall_comparison['date'], rainfall_comparison[column], marker='o', label=label)
plt.title('Rainfall forecast comparison')
plt.xlabel('Month')
plt.ylabel('Rainfall (mm)')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
temperature_comparison, temperature_metrics = run_experiment(district_weather, 'temperature_avg_c', clip_zero=False)
display(temperature_metrics)
display(temperature_comparison)

plt.figure(figsize=(14, 5))
for column, label in [('actual', 'Actual'), ('seasonal_naive', 'Seasonal Naive'), ('prophet', 'Prophet')]:
    plt.plot(temperature_comparison['date'], temperature_comparison[column], marker='o', label=label)
plt.title('Average temperature forecast comparison')
plt.xlabel('Month')
plt.ylabel('Temperature (C)')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
def rolling_origin_evaluation(data: pd.DataFrame, target: str, clip_zero: bool) -> pd.DataFrame:
    series = prepare_target(data, target)
    results = []
    for test_year in sorted(series['ds'].dt.year.unique()):
        train = series[series['ds'].dt.year < test_year].copy()
        test = series[series['ds'].dt.year == test_year].copy()
        if len(train) < MIN_TRAIN_MONTHS or len(test) < 6:
            continue
        naive = seasonal_naive_forecast(train, test)
        prophet = prophet_forecast(train, test, clip_zero)
        for model, prediction in [('Seasonal Naive', naive), ('Prophet', prophet)]:
            results.append({'target': target, 'test_year': int(test_year), 'model': model, **evaluate_forecast(test['y'], prediction)})
    return pd.DataFrame(results)

rolling_results = pd.concat([
    rolling_origin_evaluation(district_weather, 'rainfall_mm', clip_zero=True),
    rolling_origin_evaluation(district_weather, 'temperature_avg_c', clip_zero=False),
], ignore_index=True)
display(rolling_results)

In [ ]:
rainfall_metrics = rainfall_metrics.assign(target='rainfall_mm')
temperature_metrics = temperature_metrics.assign(target='temperature_avg_c')
experiment_summary = pd.concat([rainfall_metrics, temperature_metrics], ignore_index=True)
experiment_summary.insert(0, 'district_id', EXPERIMENT_DISTRICT_ID)
experiment_summary = experiment_summary[['district_id', 'target', 'model', 'MAE', 'RMSE', 'MAPE', 'n']]
display(experiment_summary)

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
experiment_summary.to_csv(RESULTS_PATH, index=False)
print('Saved:', RESULTS_PATH)

# Experiment conclusion

Complete this section after running the notebook with actual outputs.

- Dataset: monthly station-derived district observations from 2021–2025.
- Targets: monthly rainfall and monthly average temperature.
- Validation: chronological holdout and rolling-origin evaluation. Random splitting is not used.
- Model selection: choose the model with lower error against the seasonal-naive baseline for each target.
- Limitations: short historical period, unequal district coverage, missing observations, no supplied humidity data, and monthly—not daily—rainfall resolution.

Do not select Prophet merely because it runs. Report the measured MAE and RMSE, including cases where seasonal naive performs better.